In [ ]:
# Cell 1
# Install requirements

%pip install -q boto3 botocore pandas ipywidgets

In [ ]:
# Cell 2
# Imports and display settings

import os
import json
from collections import defaultdict

import boto3
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", None)

In [ ]:
# Cell 3
# Config and session setup
# Important: explicitly set Bedrock endpoints so a global AWS_ENDPOINT_URL does not misroute Bedrock to S3

REGION = os.environ.get("AWS_DEFAULT_REGION", "ca-central-1").strip()

BEDROCK_ENDPOINT = f"https://bedrock.{REGION}.amazonaws.com"
BEDROCK_RUNTIME_ENDPOINT = f"https://bedrock-runtime.{REGION}.amazonaws.com"

session = boto3.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"].strip(),
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"].strip(),
    aws_session_token=os.environ["AWS_SESSION_TOKEN"].strip(),
    region_name=REGION,
)

bedrock = session.client(
    "bedrock",
    region_name=REGION,
    endpoint_url=BEDROCK_ENDPOINT,
)

bedrock_runtime = session.client(
    "bedrock-runtime",
    region_name=REGION,
    endpoint_url=BEDROCK_RUNTIME_ENDPOINT,
)

bedrock_agent_runtime = session.client(
    "bedrock-agent-runtime",
    region_name=REGION,
)

def mask(s, left=4, right=4):
    if not s:
        return None
    if len(s) <= left + right:
        return s
    return s[:left] + "..." + s[-right:]

print("REGION:", REGION)
print("AWS_ACCESS_KEY_ID:", mask(os.environ.get("AWS_ACCESS_KEY_ID", "")))
print("AWS_ENDPOINT_URL:", os.environ.get("AWS_ENDPOINT_URL"))
print("bedrock endpoint:", bedrock.meta.endpoint_url)
print("bedrock_runtime endpoint:", bedrock_runtime.meta.endpoint_url)

In [ ]:
# Cell 4
# Inventory helper

def safe_call(name, fn):
    try:
        result = fn()
        print(f"{name}: SUCCESS")
        return result
    except Exception as e:
        print(f"{name}: FAILED")
        print(type(e).__name__, str(e))
        return None

In [ ]:
# Cell 5
# Pull inventory from Bedrock

foundation_resp = safe_call("foundation_models", lambda: bedrock.list_foundation_models())
profiles_resp = safe_call("inference_profiles", lambda: bedrock.list_inference_profiles())
provisioned_resp = safe_call("provisioned_model_throughputs", lambda: bedrock.list_provisioned_model_throughputs())
custom_models_resp = safe_call("custom_models", lambda: bedrock.list_custom_models())

foundation_models = foundation_resp.get("modelSummaries", []) if foundation_resp else []
inference_profiles = profiles_resp.get("inferenceProfileSummaries", []) if profiles_resp else []
provisioned_models = provisioned_resp.get("provisionedModelSummaries", []) if provisioned_resp else []
custom_models = custom_models_resp.get("modelSummaries", []) if custom_models_resp else []

print("\nCounts")
print("Foundation models:", len(foundation_models))
print("Inference profiles:", len(inference_profiles))
print("Provisioned throughputs:", len(provisioned_models))
print("Custom models:", len(custom_models))

In [ ]:
# Cell 6
# Show inventory tables

foundation_df = pd.DataFrame([
    {
        "modelId": m.get("modelId"),
        "provider": m.get("providerName"),
        "inputModalities": ", ".join(m.get("inputModalities", [])),
        "outputModalities": ", ".join(m.get("outputModalities", [])),
        "inferenceTypesSupported": ", ".join(m.get("inferenceTypesSupported", [])),
    }
    for m in foundation_models
]).sort_values(["provider", "modelId"], kind="stable").reset_index(drop=True)

profiles_df = pd.DataFrame([
    {
        "inferenceProfileId": p.get("inferenceProfileId"),
        "inferenceProfileName": p.get("inferenceProfileName"),
        "type": p.get("type"),
        "status": p.get("status"),
        "modelArns": ", ".join([x.get("modelArn", "") for x in p.get("models", [])]),
    }
    for p in inference_profiles
]).sort_values(["type", "inferenceProfileId"], kind="stable").reset_index(drop=True)

provisioned_df = pd.DataFrame(provisioned_models)
custom_models_df = pd.DataFrame(custom_models)

print("Foundation models")
display(foundation_df)

print("\nInference profiles")
display(profiles_df)

print("\nProvisioned model throughputs")
display(provisioned_df)

print("\nCustom models")
display(custom_models_df)

In [ ]:
# Cell 7
# Build model ARN -> inference profile map, and choose the best invoke target for each model

profiles_by_model_arn = defaultdict(list)
for profile in inference_profiles:
    if profile.get("status") != "ACTIVE":
        continue
    for model_ref in profile.get("models", []):
        model_arn = model_ref.get("modelArn")
        if model_arn:
            profiles_by_model_arn[model_arn].append(profile)

def profile_rank(profile_id: str) -> tuple:
    if profile_id.startswith("ca."):
        return (0, profile_id)
    if profile_id.startswith("us."):
        return (1, profile_id)
    if profile_id.startswith("global."):
        return (2, profile_id)
    return (3, profile_id)

def choose_target(model_summary):
    model_id = model_summary["modelId"]
    model_arn = model_summary.get("modelArn")
    inference_types = model_summary.get("inferenceTypesSupported", [])

    if "ON_DEMAND" in inference_types:
        return {
            "invoke_target": model_id,
            "invoke_target_type": "MODEL_ID",
            "profile_used": None,
        }

    if "INFERENCE_PROFILE" in inference_types:
        profiles = sorted(
            profiles_by_model_arn.get(model_arn, []),
            key=lambda p: profile_rank(p["inferenceProfileId"]),
        )
        if not profiles:
            return {
                "invoke_target": None,
                "invoke_target_type": "INFERENCE_PROFILE",
                "profile_used": None,
            }
        chosen = profiles[0]
        return {
            "invoke_target": chosen["inferenceProfileId"],
            "invoke_target_type": "INFERENCE_PROFILE",
            "profile_used": chosen["inferenceProfileId"],
        }

    return {
        "invoke_target": model_id,
        "invoke_target_type": "MODEL_ID",
        "profile_used": None,
    }

targets_df = pd.DataFrame([
    {
        "modelId": m["modelId"],
        "provider": m.get("providerName"),
        "inferenceTypesSupported": ", ".join(m.get("inferenceTypesSupported", [])),
        **choose_target(m),
    }
    for m in foundation_models
]).sort_values(["provider", "modelId"], kind="stable").reset_index(drop=True)

display(targets_df)

In [ ]:
# Cell 8
# Smoke test helpers

def parse_text_response(resp):
    chunks = resp["output"]["message"]["content"]
    parts = [c.get("text", "") for c in chunks if isinstance(c, dict) and "text" in c]
    return "".join(parts).strip()

def converse_smoke_test(target):
    resp = bedrock_runtime.converse(
        modelId=target,
        messages=[
            {
                "role": "user",
                "content": [{"text": "Reply with exactly: ok"}],
            }
        ],
        inferenceConfig={
            "maxTokens": 10,
            "temperature": 0.0,
        },
    )
    return parse_text_response(resp)

def embed_smoke_test(model_summary, target):
    model_id = model_summary["modelId"].lower()

    if model_id.startswith("amazon.titan-embed"):
        body = {"inputText": "hello"}
    elif model_id.startswith("cohere.embed"):
        body = {
            "texts": ["hello"],
            "input_type": "search_document",
        }
    else:
        raise RuntimeError("No generic embedding payload configured for this model")

    resp = bedrock_runtime.invoke_model(
        modelId=target,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json",
    )
    payload = json.loads(resp["body"].read())

    if "embedding" in payload and isinstance(payload["embedding"], list):
        return f"embedding_length={len(payload['embedding'])}"

    if "embeddings" in payload and isinstance(payload["embeddings"], list):
        first = payload["embeddings"][0] if payload["embeddings"] else []
        if isinstance(first, list):
            return f"embedding_length={len(first)}"
        return "embeddings returned"

    return "embedding returned"

def classify_and_test_model(model):
    model_id = model["modelId"]
    provider = model.get("providerName")
    inputs = model.get("inputModalities", [])
    outputs = model.get("outputModalities", [])
    inference_types = model.get("inferenceTypesSupported", [])

    target_info = choose_target(model)
    target = target_info["invoke_target"]

    row = {
        "modelId": model_id,
        "provider": provider,
        "inputModalities": ", ".join(inputs),
        "outputModalities": ", ".join(outputs),
        "inferenceTypesSupported": ", ".join(inference_types),
        "invoke_target": target,
        "invoke_target_type": target_info["invoke_target_type"],
        "profile_used": target_info["profile_used"],
        "status": None,
        "detail": None,
    }

    try:
        if target is None:
            row["status"] = "NO_PROFILE_FOUND"
            row["detail"] = "Model requires an inference profile, but no active matching profile was found."
        elif "EMBEDDING" in outputs:
            row["status"] = "WORKS"
            row["detail"] = embed_smoke_test(model, target)
        elif model_id.lower().startswith("amazon.rerank") or model_id.lower().startswith("cohere.rerank"):
            row["status"] = "SKIPPED"
            row["detail"] = "Rerank model, not tested with generic text-generation smoke test."
        elif "VIDEO" in inputs and "TEXT" in outputs and "TEXT" not in inputs:
            row["status"] = "SKIPPED"
            row["detail"] = "Video-input model, not tested with text-only smoke test."
        elif "TEXT" in outputs and "TEXT" in inputs:
            row["status"] = "WORKS"
            row["detail"] = converse_smoke_test(target)
        else:
            row["status"] = "SKIPPED"
            row["detail"] = "No generic smoke test configured for this modality combination."
    except Exception as e:
        row["status"] = "FAILS"
        row["detail"] = f"{type(e).__name__}: {e}"

    return row

In [ ]:
# Cell 9
# Run smoke test across all listed foundation models and save results

results = [classify_and_test_model(model) for model in foundation_models]

results_df = pd.DataFrame(results).sort_values(
    by=["status", "provider", "modelId"],
    ascending=[True, True, True],
    kind="stable",
).reset_index(drop=True)

display(results_df)

results_df.to_csv("bedrock_model_smoke_test_results.csv", index=False)
foundation_df.to_csv("bedrock_foundation_models.csv", index=False)
profiles_df.to_csv("bedrock_inference_profiles.csv", index=False)

print("Saved:")
print(" - bedrock_model_smoke_test_results.csv")
print(" - bedrock_foundation_models.csv")
print(" - bedrock_inference_profiles.csv")

In [ ]:
# Cell 10
# Quick summary tables

print("Smoke test status counts")
display(results_df.groupby("status").size().reset_index(name="count"))

print("\nModels that worked")
display(results_df.loc[results_df["status"] == "WORKS", [
    "modelId", "provider", "invoke_target", "invoke_target_type", "detail"
]].reset_index(drop=True))

print("\nModels that failed")
display(results_df.loc[results_df["status"] == "FAILS", [
    "modelId", "provider", "invoke_target", "invoke_target_type", "detail"
]].reset_index(drop=True))

print("\nModels skipped or requiring special handling")
display(results_df.loc[results_df["status"].isin(["SKIPPED", "NO_PROFILE_FOUND"]), [
    "modelId", "provider", "invoke_target", "invoke_target_type", "detail"
]].reset_index(drop=True))

In [ ]:
# Cell 11
# Example 1
# Direct text or chat model, ON_DEMAND
# Good for models like:
# - anthropic.claude-3-haiku-20240307-v1:0
# - anthropic.claude-3-sonnet-20240229-v1:0
# - meta.llama3-8b-instruct-v1:0
# - meta.llama3-70b-instruct-v1:0
# - mistral.mistral-7b-instruct-v0:2
# - mistral.mixtral-8x7b-instruct-v0:1
# - mistral.mistral-large-2402-v1:0

model_id = "meta.llama3-8b-instruct-v1:0"

response = bedrock_runtime.converse(
    modelId=model_id,
    messages=[
        {
            "role": "user",
            "content": [{"text": "Reply with exactly: bedrock direct model works"}],
        }
    ],
    inferenceConfig={
        "maxTokens": 40,
        "temperature": 0.0,
    },
)

print(parse_text_response(response))

In [ ]:
# Cell 12
# Example 2
# Text or chat model that must be invoked through an inference profile
# Good for models like:
# - amazon.nova-lite-v1:0, use ca.amazon.nova-lite-v1:0
# - amazon.nova-2-lite-v1:0, use global.amazon.nova-2-lite-v1:0 or us.amazon.nova-2-lite-v1:0
# - anthropic.claude-sonnet-4-6, use us.anthropic.claude-sonnet-4-6 or global.anthropic.claude-sonnet-4-6

profile_id = "ca.amazon.nova-lite-v1:0"

response = bedrock_runtime.converse(
    modelId=profile_id,
    messages=[
        {
            "role": "user",
            "content": [{"text": "Reply with exactly: bedrock inference profile works"}],
        }
    ],
    inferenceConfig={
        "maxTokens": 40,
        "temperature": 0.0,
    },
)

print(parse_text_response(response))

In [ ]:
# Cell 13
# Example 3
# Embedding model, direct ON_DEMAND
# Good for:
# - amazon.titan-embed-text-v2:0
# - amazon.titan-embed-image-v1
# - cohere.embed-english-v3
# - cohere.embed-multilingual-v3
#
# Note:
# - Titan uses {"inputText": "..."}
# - Cohere embed models use {"texts": [...], "input_type": "search_document"}

embed_model_id = "amazon.titan-embed-text-v2:0"

body = {"inputText": "hello from Bedrock embeddings"}

response = bedrock_runtime.invoke_model(
    modelId=embed_model_id,
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json",
)

payload = json.loads(response["body"].read())
print(payload.keys())
print("embedding length:", len(payload["embedding"]))

In [ ]:
# Cell 14
# Example 4
# Embedding model via inference profile
# Good for:
# - cohere.embed-v4:0, use global.cohere.embed-v4:0

embed_profile_id = "global.cohere.embed-v4:0"

body = {
    "texts": ["hello from Bedrock Cohere Embed v4"],
    "input_type": "search_document",
}

response = bedrock_runtime.invoke_model(
    modelId=embed_profile_id,
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json",
)

payload = json.loads(response["body"].read())
print(payload.keys())
print(payload)

In [ ]:
# Cell 15
# Example 5
# Rerank model
# Good for:
# - amazon.rerank-v1:0
# - cohere.rerank-v3-5:0
#
# This uses the Bedrock Agent Runtime rerank API, not converse()

rerank_model_arn = f"arn:aws:bedrock:{REGION}::foundation-model/amazon.rerank-v1:0"

rerank_response = bedrock_agent_runtime.rerank(
    queries=[
        {
            "type": "TEXT",
            "textQuery": {
                "text": "Which document is about model deployment?"
            },
        }
    ],
    sources=[
        {
            "type": "INLINE",
            "inlineDocumentSource": {
                "type": "TEXT",
                "textDocument": {
                    "text": "This note describes how to deploy an LLM endpoint with Bedrock.",
                },
            },
        },
        {
            "type": "INLINE",
            "inlineDocumentSource": {
                "type": "TEXT",
                "textDocument": {
                    "text": "This note is about woodworking and cabinet face frames.",
                },
            },
        },
    ],
    rerankingConfiguration={
        "type": "BEDROCK_RERANKING_MODEL",
        "bedrockRerankingConfiguration": {
            "modelConfiguration": {
                "modelArn": rerank_model_arn
            },
            "numberOfResults": 2,
        },
    },
)

print(json.dumps(rerank_response, indent=2, default=str))

In [ ]:
# Cell 16
# Example 6
# Pick any row from the smoke test table and invoke it using the target the table chose

example_row = results_df.loc[results_df["status"] == "WORKS"].iloc[0]
example_target = example_row["invoke_target"]
example_model_id = example_row["modelId"]

print("Chosen modelId:", example_model_id)
print("Chosen invoke target:", example_target)

if "EMBEDDING" in example_row["outputModalities"]:
    lower_id = example_model_id.lower()
    if lower_id.startswith("amazon.titan-embed"):
        body = {"inputText": "test embedding"}
    elif lower_id.startswith("cohere.embed"):
        body = {"texts": ["test embedding"], "input_type": "search_document"}
    else:
        raise RuntimeError("No example payload configured for this embedding model")

    response = bedrock_runtime.invoke_model(
        modelId=example_target,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json",
    )
    payload = json.loads(response["body"].read())
    print(payload)
else:
    response = bedrock_runtime.converse(
        modelId=example_target,
        messages=[
            {
                "role": "user",
                "content": [{"text": "Reply with exactly: example target works"}],
            }
        ],
        inferenceConfig={"maxTokens": 40, "temperature": 0.0},
    )
    print(parse_text_response(response))